In [11]:
from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path
import shutil
import torch
import os
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise
from datasets import Dataset, Image, load_dataset

In [ ]:
# from huggingface_hub import snapshot_download
# import zipfile
# from pathlib import Path

# # --- Configuration ---
# REPO_ID = "lgrzybowski/seraphim-drone-detection-dataset"
# LOCAL_DIR = Path("repository_location") # TODO: change to your local directory

# # --- Step 1: Download the entire repo ---
# repo_path = Path(snapshot_download(repo_id=REPO_ID, repo_type="dataset", local_dir=LOCAL_DIR))

# # --- Step 2: Unzip all .zip files in place ---
# zip_files = list(repo_path.rglob("*.zip"))
# print(f"Found {len(zip_files)} zip files to extract")

# for zip_path in zip_files:
#     try:
#         with zipfile.ZipFile(zip_path, "r") as z:
#             z.extractall(zip_path.parent)
#         print(f"✅ Extracted: {zip_path.relative_to(repo_path)}")
#         zip_path.unlink()  # remove the zip file
#     except zipfile.BadZipFile:
#         print(f"⚠️ Skipping invalid zip: {zip_path}")

# print("🎉 All zips extracted and removed.")
# print(f"📂 Dataset ready at: {repo_path.resolve()}")


/home/stevensagun/projects/DS681-Assignment-3/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 17 files: 100%|██████████| 17/17 [01:44<00:00,  6.13s/it]


Found 7 zip files to extract
✅ Extracted: train/images/batch_003.zip
✅ Extracted: train/images/batch_002.zip
✅ Extracted: train/images/batch_001.zip
✅ Extracted: train/images/batch_004.zip
✅ Extracted: train/labels/batch_001.zip
✅ Extracted: test/images/batch_001.zip
✅ Extracted: test/labels/batch_001.zip
🎉 All zips extracted and removed.
📂 Dataset ready at: /home/stevensagun/projects/DS681-Assignment-3/repository_location


In [ ]:
# model = YOLO("yolov8m.pt")
# model.train(data="./drone.yaml", epochs=5, imgsz=640)
# model.save("model_5.pt")

Ultralytics 8.4.19 🚀 Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./drone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0

In [6]:
model = YOLO("model_5.pt")

In [ ]:

def remove_dir(directory_path: str):
    detections_directory = Path(directory_path)
    if detections_directory.exists() and detections_directory.is_dir():
        shutil.rmtree(detections_directory)
        print("Directory deleted")
    else:
        print("Directory does not exist")

# Delete output folder if it exists.
remove_dir('./detections')
remove_dir('./outputs')
os.makedirs('./detections')
os.makedirs('./outputs')

class BoundingBox:
    def __init__(self, x1: int, y1: int, x2: int, y2: int):
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

class Center:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

class FrameData:
    def __init__(self, is_detection: bool, bounding_box: BoundingBox, center: Center):
        self.is_detection = is_detection
        self.bounding_box = bounding_box
        self.center = center

class OverlayFrameData:
    def __init__(self, bounding_box: BoundingBox | None, center: Center):
        # The bounding box of the detection
        self.bounding_box = bounding_box
        # The predicted trajectory point
        self.center = center


frame_interval = 0
height = 0
width = 0
def extract_detected_frames(name: str):
    global frame_interval
    global height
    global width
    video_path = f'./videos/{name}'
    cap = cv2.VideoCapture(video_path)
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(original_fps/ 5)
    frame_count = 0
    frame_detection_data: list[FrameData | None] = []
    initialized_dimensions = False
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if initialized_dimensions is False:
            height, width = frame.shape[:2]
            initialized_dimensions = True

        # Just initialize it to None for this frame
        frame_detection_data.append(None)

        if frame_count % frame_interval == 0 :
            results = model.predict(frame)
            
            if results[0].boxes.conf.numel() > 0:
                index = torch.argmax(results[0].boxes.conf) 
                if results[0].boxes.conf[index] >= 0.7:
                    x1, y1, x2, y2 = results[0].boxes.xyxy[index].round().int().tolist()
                    x, y, _w, _h = results[0].boxes.xywh[index].round().int().tolist()
                    bounding_box = BoundingBox(x1, y1, x2, y2)
                    center = Center(x, y)
                    detection = FrameData(True, bounding_box, center)
                    frame_detection_data[-1] = detection

                    # For part 1 deliverable
                    cv2.rectangle(
                        frame,
                        (x1, y1),
                        (x2, y2),
                        (0, 0, 255),
                        2             # Thickness
                    )
                    name_without_ext = os.path.splitext(name)[0]
                    os.makedirs(f'./detections/{name_without_ext}', exist_ok=True)
                    cv2.imwrite(f'./detections/{name_without_ext}/{frame_count}.jpg', frame)

        frame_count += 1

    cap.release()
    return frame_detection_data     
                
def is_out_of_bounds(center: Center):
    return center.x < 0 or center.x >= width or center.y < 0 or center.y >= height

def create_kalman_filter() -> KalmanFilter:
    dt = 1/30
    variance = 5
    process_var = 250
    # dim_x = 4 for position (x, y) and veloicy x and y. dim_z for the known position
    kalman_filter = KalmanFilter(dim_x=6, dim_z=2)
    # x_pred = Fx. x is dim_x ^. [x, y, velocity_x, velocity_y]

    dt2 = dt * dt
    kalman_filter.F = np.array([
        [1, 0, dt, 0, 0.5 * dt2, 0],
        [0 , 1, 0, dt, 0, 0.5 * dt],
        [0, 0, 1, 0, dt, 0],
        [0, 0, 0, 1, 0, dt],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1]
    ], dtype=float)
    #measurement. we only know the x and y position
    kalman_filter.H = np.array([
        [1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0]
    ], dtype=float)
    kalman_filter.R = np.eye(2) * variance

    q = Q_discrete_white_noise(dim=3, dt=dt, var=process_var)
    kalman_filter.Q = np.block([
        [q, np.zeros((3,3))],
        [np.zeros((3,3)), q]
    ])

    # We're pretty certian for the x and y but not certain of velocity or acceleration
    kalman_filter.P = np.diag([25, 25, 1000, 1000, 10000, 10000]).astype(float)

    kalman_filter.x = np.zeros((6, 1), dtype=float)

    return kalman_filter

def predict_trajectory(detection_frame_data: list[FrameData | None]) -> list[OverlayFrameData | None]:
    overlay_frame_data = []
    kalman_filter = None
    init = False
    out_of_bounds = False

    for frame_data in detection_frame_data:

        if init is False and frame_data is None:
            overlay_frame_data.append(None)
            continue
        elif init is False and frame_data is not None:
            kalman_filter = create_kalman_filter()
            x = frame_data.center.x
            y = frame_data.center.y
             # Initialize the filter. 0 for velocity
            kalman_filter.x = np.array([[x], [y], [0.0], [0.0], [0.0], [0.0]])
            init = True
            bbox = BoundingBox(frame_data.bounding_box.x1, frame_data.bounding_box.y1, frame_data.bounding_box.x2, frame_data.bounding_box.y2)
            center = Center(x, y)
            overlay_frame_data.append(OverlayFrameData(bbox, center))
            continue

        # Normal case
        kalman_filter.predict()

        bbox = None


        if frame_data is not None:
            # In case the trajectory was out of bounds in the previous frame, initialize a new filter
            if out_of_bounds:
                kalman_filter = create_kalman_filter()
                x = frame_data.center.x
                y = frame_data.center.y
                # Initialize the filter. 0 for velocity
                kalman_filter.x = np.array([[x], [y], [0.0], [0.0], [0.0], [0.0]])
                init = True
                bbox = BoundingBox(frame_data.bounding_box.x1, frame_data.bounding_box.y1, frame_data.bounding_box.x2, frame_data.bounding_box.y2)
                center = Center(x, y)
                overlay_frame_data.append(OverlayFrameData(bbox, center))

                out_of_bounds = False
                continue

            bbox = frame_data.bounding_box
            z = np.array([[frame_data.center.x], [frame_data.center.y]])
            kalman_filter.update(z)
            
        predict_x, predict_y, _velocity_x, _velocity_y, _acc_x, _acc_y = kalman_filter.x.flatten()
        center = Center(int(round(predict_x)), int(round(predict_y)))
        out_of_bounds = is_out_of_bounds(center)
        if out_of_bounds:
            # Signals to reset poly line
            overlay_frame_data.append(True)
            continue

        overlay_frame_data.append(OverlayFrameData(BoundingBox(bbox.x1, bbox.y1, bbox.x2, bbox.y2) if bbox is not None else None, center))

    return overlay_frame_data

def create_output(name: str, overlay_frame_data: list[OverlayFrameData | None]):
    video_path = f'./videos/{name}'
    output_path = f'./outputs/output_{name}'
    os.makedirs(f'./outputs', exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_index = 0
    polyline = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        out_frame = frame

        frame_data = overlay_frame_data[frame_index]

        if frame_data is True:
            polyline = []
        elif frame_data is not None:
            out_frame = frame.copy()
            if frame_data.bounding_box is not None:
                bbox = frame_data.bounding_box
                cv2.rectangle(out_frame, (bbox.x1, bbox.y1), (bbox.x2, bbox.y2), (0, 0, 255), 2)
            polyline.append((frame_data.center.x, frame_data.center.y))
            pts = np.array(polyline, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(out_frame, [pts], isClosed=False, color=(0, 255, 0), thickness=1)

        writer.write(out_frame)
        frame_index += 1
    cap.release()
    writer.release()

### This process each video inside the videos folder.
directory = Path("./videos")
overlay_frame_data = None
for file in directory.iterdir():
    if file.is_file():
        detection_frame_data = extract_detected_frames(file.name)
        overlay_frame_data = predict_trajectory(detection_frame_data)       
        create_output(file.name, overlay_frame_data)


    

Directory deleted
Directory deleted

0: 384x640 (no detections), 18.9ms
Speed: 1.7ms preprocess, 18.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 19.6ms
Speed: 1.5ms preprocess, 19.6ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 26.1ms
Speed: 1.6ms preprocess, 26.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 24.2ms
Speed: 1.5ms preprocess, 24.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 22.1ms
Speed: 1.4ms preprocess, 22.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 32.8ms
Speed: 1.5ms preprocess, 32.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 17.5ms
Speed: 1.6ms preprocess, 17.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 drone, 18.1ms
Speed

In [15]:
import pandas as pd
# Create Parquet
detections_dir = Path("./detections")

rows = []

for img_path in detections_dir.rglob("*.jpg"):

    rel_path = img_path.relative_to(detections_dir)

    rows.append({
        "image": str(rel_path),        # relative path
        "frame": int(img_path.stem),
        "video": img_path.parent.name
    })

df = pd.DataFrame(rows)

df.to_parquet("./detections/detections.parquet", engine="pyarrow")
